In [ ]:
# --- cell 1 · bootstrap（任务书 §0.1 末、§0.2）—— 五个 B notebook 里这一格完全相同 ---
import pathlib, sys

# (1) 向上搜索含 npnn/ 的目录 —— 不依赖 Jupyter 的启动目录
def _find_taskb_root(start):
    for p in (start, *start.parents):
        if (p / 'npnn').is_dir():
            return p
        if (p / 'task_b' / 'npnn').is_dir():      # 从项目根启动 Jupyter 的情况
            return p / 'task_b'
    return None

_here = pathlib.Path.cwd().resolve()
TASKB_ROOT = _find_taskb_root(_here)
assert TASKB_ROOT is not None, f'从 {_here} 逐级往上找不到含 npnn/ 的目录；请从 task_b/ 或项目根启动'
sys.path.insert(0, str(TASKB_ROOT))

# (2) 物理挡住「误 import A 的 src」：把项目根从 sys.path 里摘掉（任务书 §0.1.2）
_root = TASKB_ROOT.parent
sys.path[:] = [e for e in sys.path if pathlib.Path(e or '.').resolve() != _root]

# (3) 线程预算：必须在 import numpy 之前（BLAS 首次 import 时锁死线程数）
import config
config.limit_threads()

# (4) 环境留痕 —— 报告 B.1 要用：哪个解释器、哪个 numpy
import numpy as np
print('sys.executable:', sys.executable)
print('sys.prefix    :', sys.prefix)
print('numpy         :', np.__version__)
print('TASKB_ROOT    :', TASKB_ROOT)


# 12 · 与 PyTorch 整网对拍（L2 —— 10 分的核心证据）

对应任务书：§3.1（L2 比 L3 重要得多）、§3.2（严格协议）。

> L3 的「精度相近」是弱证据（lr 抖动、随机性都能造成几个点差异）；
> L2 的「loss 轨迹逐位重合」是强证据 —— 它证明整个前向 + 反向都对。

## §3.2 协议（照着做）
| 环节 | 做法 | 坑 |
|---|---|---|
| 架构 | 定义**一次**（`NET` 列表），两边用同一份配置生成 | 别一边 3 层一边 4 层 |
| 权重 | numpy 侧生成 → `copy_()` 按顺序写入，带形状断言 | dtype / 形状要一致 |
| 数据 | 两边拿到**同一个** float64 归一化 batch | 归一化公式必须一致 |
| 优化器 | 我们的 Adam vs `torch.optim.Adam`，同 lr / betas / eps | 见 §2.4 四坑 |
| 设备 | **全程 CPU + float64** | GPU/cuDNN 非确定性会掩盖真 bug |
| 判据 | N 步后 loss 相对差 <1e-8 | 分叉 → 二分定位 |

## 本 notebook 的产出
- [x] **cell 2**：跑 `tests/l2_parity.py` —— 100 步、batch 32、lr 1e-3、float64
      → `max|Δloss| = 1.776e-15`，`max|Δparam| = 4.0e-15`，**全程没有一步超过 1e-10**
- [x] **cell 3**：报告图 —— loss 轨迹重合 + `|Δloss|` 对数曲线 + 最终参数差直方图
      （`report/figs/l2_parity.png`, `l2_param_hist.png`）

⚠️ torch 没有 SwitchNorm2d，参照实现是 `TorchSN`（torch 张量 + autograd 写的同一个公式，
与 numpy 版不共享一行代码）—— 这反而是更强的证据：两条独立实现走出了同一条轨迹。

⛔ 本 notebook 可以 import torch —— 它是验证工具，不是交付实现（报告里要明确区分）。

In [ ]:
# --- cell 2 · L2 整网对拍（脚本 tests/l2_parity.py）---
# 同一份权重 / 同一批数据 / 同一套超参，numpy 与 torch 各跑 100 步，比较两条 loss 轨迹
import subprocess, sys

r = subprocess.run([sys.executable, str(TASKB_ROOT / 'tests' / 'l2_parity.py')],
                   capture_output=True, text=True, encoding='utf-8', errors='replace')
print(r.stdout)
print('exit code:', r.returncode, '（0 = PASS）')

In [ ]:
# --- cell 3 · 报告图：loss 轨迹重合 + |Δloss| 对数曲线 + 最终参数差直方图 ---
import json
import matplotlib.pyplot as plt
from config import FIG_DIR, RUNS_DIR

p = json.loads((RUNS_DIR / 'l2' / 'parity.json').read_text(encoding='utf-8'))
steps = np.arange(1, len(p['loss_np']) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.9))
ax1.plot(steps, p['loss_np'], label='numpy', lw=1.8)
ax1.plot(steps, p['loss_t'], '--', label='torch', lw=1.2)
ax1.set_xlabel('step'); ax1.set_ylabel('training loss')
ax1.set_title('same weights, data and optimizer')
ax1.legend(); ax1.grid(alpha=0.3)

ax2.semilogy(steps, np.maximum(p['loss_diff'], 1e-18), lw=1.4)
ax2.axhline(1e-8, color='tab:green', ls='--', lw=1, label='target 1e-8')
ax2.set_xlabel('step'); ax2.set_ylabel(r'$|\Delta\mathrm{loss}|$')
ax2.set_title('difference (log scale)')
ax2.legend(); ax2.grid(alpha=0.3, which='both')
fig.tight_layout()
fig.savefig(FIG_DIR / 'l2_parity.png', dpi=150, bbox_inches='tight')
plt.show()

fig2, ax = plt.subplots(figsize=(6, 3.6))
h = p['param_hist']
ax.bar(h['edges'][:-1], h['counts'], width=np.diff(h['edges']), align='edge',
       color='tab:blue', alpha=0.85)
ax.set_yscale('log')
ax.set_xlabel(r'$|\Delta\theta|$ after 100 steps'); ax.set_ylabel('count')
ax.set_title(f'final parameter difference ({p["n_params"]} params)')
fig2.tight_layout()
fig2.savefig(FIG_DIR / 'l2_param_hist.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'max |Δloss| = {p["max_loss_diff"]:.3e}   (target < 1e-8)')
print(f'first step with |Δloss| > 1e-10: {p["first_divergence_step"]}')